## Conversion from Keras HGQ2 model to HLS4ML model (Verilog)

Convert the quantized model to HLS (High-Level Synthesis) code for FPGA deployment. Precision is automatically inferred from the HGQ2 model.

In [ ]:
import os, subprocess
import shutil
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Keras 3
import keras

# Package imports
from neuroposasic.data import load_dataset, preprocess_data

import hls4ml
from hls4ml.model.profiling import numerical, compare

from sklearn.metrics import log_loss, accuracy_score # sklearn has an easy interface for testing
from scipy.special import softmax # log_loss requires probabilities not found in HLS because 'from_logits=True'

PROFILE = True
BUILD_IP = True
BUILD_PYNQ = True
RESET_BUILD = True

In [ ]:
# Configure matplotlib for PDF export with serif font
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Computer Modern Serif",
})

# Ensure figures directory exists
os.makedirs('figures', exist_ok=True)

# Set random seed for reproducibility
np.random.seed(42)
keras.utils.set_random_seed(42)

# Constant number of postures
NUM_CLASSES = 3

## Data

### Loading

In [ ]:
# Load datasets using package function
X_train, y_train, X_val, y_val, X_test, y_test = load_dataset(type="raw")

### Preprocessing

In [ ]:
# Preprocess data using package function
(X_train, y_train, X_val, y_val, X_test, y_test, class_names) = preprocess_data(
    X_train, y_train, X_val, y_val, X_test, y_test,
    num_classes=NUM_CLASSES,
)

## Model Loading

In [ ]:
model_path = "output/posture_classifier_qat.keras"
model_qat = keras.models.load_model(model_path)
model_qat.summary()

## Conversion

In [ ]:
# Make Vitis available in path
script = "/home/jeison/AMD/2025.2/Vitis/settings64.sh"

for line in subprocess.check_output(f'source "{script}" >/dev/null && env', shell=True, executable='/bin/bash', text=True).splitlines():
    if '=' in line:
        k, v = line.split('=', 1)
        os.environ[k] = v

## HLS4ML

In [ ]:
config = hls4ml.utils.config_from_keras_model(model_qat, granularity='name')

config['Model']['Strategy'] = 'Resource'
# config['Model']['ReuseFactor'] = 2
config['Model']['IOType'] = 'io_stream'

config['LayerName']['qconv_1']['ReuseFactor'] = 9
config['LayerName']['qconv_1']['Trace'] = True
config['LayerName']['qconv_2']['ReuseFactor'] = 108
config['LayerName']['qconv_2']['Trace'] = True
config['LayerName']['qconv_3']['ReuseFactor'] = 108
config['LayerName']['qconv_3']['Trace'] = True

config['LayerName']['output']['ReuseFactor'] = 60
config['LayerName']['output']['Trace'] = True
# config['LayerName']['output']['Strategy'] = 'Latency'

config['LayerName']['bn_conv_1']['Precision'] = 'ap_fixed<8,4>'
config['LayerName']['bn_conv_2']['Precision'] = 'ap_fixed<8,4>'
config['LayerName']['bn_conv_3']['Precision'] = 'ap_fixed<8,4>'
config['LayerName']['bn_dense_1']['Precision'] = 'ap_fixed<8,4>'
config['LayerName']['bn_dense_2']['Precision'] = 'ap_fixed<8,4>'


# Convert model
hls_model = hls4ml.converters.convert_from_keras_model(
    model_qat,
    hls_config=config,
    output_dir='../soft_ip',
    backend='Vitis',
    part='xc7z020clg400-1', # Pynq-Z2
    clock_period=10,
    clock_uncertainty='12.5%',
)

## Profiling

In [ ]:
# Compile and synthesize the model
hls_model.compile()

# Shows the median and quartiles of weight/activation distributions with box-and-whisker diagrams
# Grey box should cover the full range of the box-and-whisker plot

if PROFILE:
    wp, wph, ap, aph = numerical(
        model=model_qat,
        hls_model=hls_model,
        X=X_test,
        plot='boxplot'
    )

    compare(
        model_qat,
        hls_model,
        X_test,
        plot_type='dist_diff'
)

### Model

In [ ]:
hls4ml.utils.plot_model(hls_model, to_file="figures/hls_model.png", show_shapes=True, show_precision=True, dpi=300)

plt.figure(figsize=(12, 12))
plt.imshow(plt.imread("figures/hls_model.png"))
plt.axis("off")
plt.show()

### Evaluation

In [ ]:
# From part4.1_HG_quantization and larger_jet_tagger
X_test = np.ascontiguousarray(X_test)

# From Scr_2_HLS4ML_Vivado_generator
y_hls = hls_model.predict(X_test) # Executes the FPGA firmware with bit-accurate emulation on the CPU.
y_hls_prob = softmax(y_hls, axis=1) # Convert logits to probabilities for log_loss

# Calculating accuracy and crossentropy loss of HLS_model
hls_accuracy = accuracy_score(np.argmax(y_test, axis=1), np.argmax(y_hls, axis=1))
hls_crossentropyloss = log_loss(y_true=y_test, y_pred=y_hls_prob)
metrics_tuple = (hls_crossentropyloss, hls_accuracy)

## Determine accuracy of Keras model, for comparison
keras_score = model_qat.evaluate(X_test, y_test, verbose=0)

print("="*80)
print("HLS4ML model evaluation \t Loss: {:.2f} \t Accuracy: {:.2f}%".format(metrics_tuple[0], metrics_tuple[1]*100))    
print("Keras  model evaluation \t Loss: {:.2f} \t Accuracy: {:.2f}%".format(keras_score[0], keras_score[1]*100))
print("="*80)

In [ ]:
# Run prediction
y_keras = model_qat.predict(X_test, batch_size=2048)

# Test for mismatches
# HGQ2 and hls4ml should produce the same output, up to machine precision
# Notice that due to the quantization, internal mismatches may be amplified, but the vast majority of the output should match
print(f"{np.sum(y_keras != y_hls)} / {np.prod(y_keras.shape)} value mismatches")

## Building

### Soft IP

In [ ]:
# Build with various options
if BUILD_IP:
    report = hls_model.build(
        reset=RESET_BUILD,       # Reset project before build
        csim=True,          # Run C simulation
        synth=True,         # Run HLS synthesis
        # cosim=True,       # Run co-simulation
        # validation=False, # Run validation
        export=True,        # Export IP
        vsynth=False,       # Run Vivado synthesis
        fifo_opt=True       # FIFO optimization
    )

### Report

In [ ]:
if BUILD_IP:
    # Latency (cycles)
    best_latency = int(report['CSynthesisReport']['BestLatency'])
    worst_latency = int(report['CSynthesisReport']['WorstLatency'])
    print(f"Latency: {best_latency} - {worst_latency} cycles")

    # Clock (ns)
    target_clock = float(report['CSynthesisReport']['TargetClockPeriod'])
    estimated_clock = float(report['CSynthesisReport']['EstimatedClockPeriod'])
    print(f"Target Clock = {target_clock} ns | Estimated Clock = {estimated_clock} ns")

    # Available resources
    avail_LUT = int(report['CSynthesisReport']['AvailableLUT'])
    avail_FF = int(report['CSynthesisReport']['AvailableFF'])
    avail_DSP = int(report['CSynthesisReport']['AvailableDSP'])
    avail_BRAM = int(report['CSynthesisReport']['AvailableBRAM_18K'])

    # Used resources (Vitis HLS)
    used_LUT = int(report['CSynthesisReport']['LUT'])
    used_FF = int(report['CSynthesisReport']['FF'])
    used_DSP = int(report['CSynthesisReport']['DSP'])
    used_BRAM = int(report['CSynthesisReport']['BRAM_18K'])

    # Calculate percentages
    used_LUT_pct = (used_LUT / avail_LUT) * 100
    used_FF_pct = (used_FF / avail_FF) * 100
    used_DSP_pct = (used_DSP / avail_DSP) * 100
    used_BRAM_pct = (used_BRAM / avail_BRAM) * 100

    # Vitis HLS Resources
    print(f"\n--- Vitis HLS Resource Utilization ---")
    print(f"LUT:  {used_LUT:>6} / {avail_LUT:>6} ({used_LUT_pct:>5.2f}%)")
    print(f"FF:   {used_FF:>6} / {avail_FF:>6} ({used_FF_pct:>5.2f}%)")
    print(f"DSP:  {used_DSP:>6} / {avail_DSP:>6} ({used_DSP_pct:>5.2f}%)")
    print(f"BRAM: {used_BRAM:>6} / {avail_BRAM:>6} ({used_BRAM_pct:>5.2f}%)")

    # Used resources (Vivado)
    used_LUT_v = int(report['VivadoSynthReport']['LUT'])
    used_FF_v = int(report['VivadoSynthReport']['FF'])
    used_DSP_v = int(report['VivadoSynthReport']['DSP48E'])
    used_BRAM_v = int(report['VivadoSynthReport']['BRAM_18K'])

    # Calculate percentages
    used_LUT_v_pct = (used_LUT_v / avail_LUT) * 100
    used_FF_v_pct = (used_FF_v / avail_FF) * 100
    used_DSP_v_pct = (used_DSP_v / avail_DSP) * 100
    used_BRAM_v_pct = (used_BRAM_v / avail_BRAM) * 100

    # Vivado Synthesis Resources
    print(f"\n--- Vivado Synthesis Resource Utilization ---")
    print(f"LUT:  {used_LUT_v:>6} / {avail_LUT:>6} ({used_LUT_v_pct:>5.2f}%)")
    print(f"FF:   {used_FF_v:>6} / {avail_FF:>6} ({used_FF_v_pct:>5.2f}%)")
    print(f"DSP:  {used_DSP_v:>6} / {avail_DSP:>6} ({used_DSP_v_pct:>5.2f}%)")
    print(f"BRAM: {used_BRAM_v:>6} / {avail_BRAM:>6} ({used_BRAM_v_pct:>5.2f}%)")

### Pynq-Z2

In [ ]:
# Build manually bitstream for pynq-z2
if BUILD_PYNQ:
    repo_root = Path.cwd().parent
    vitis_dir = repo_root / 'vitis_pynq'
    vivado_dir = repo_root / 'vivado_pynq'

    # Clean previous build artifacts if requested
    if RESET_BUILD:
        for build_dir in [
            vitis_dir / 'myproject_axi_prj',
            vivado_dir / 'myproject_vivado_accelerator',
        ]:
            if build_dir.exists():
                print(f'Removing {build_dir}')
                shutil.rmtree(build_dir)

    # Step 1: Synthesize AXI-Stream wrapper with Vitis HLS
    print('--- Running Vitis HLS wrapper synthesis ---')
    hls_result = subprocess.run(
        ['vitis-run', '--mode', 'hls', '--tcl', str(vitis_dir / 'run_hls.tcl')],
        capture_output=True,
        text=True,
        cwd=str(vitis_dir),
    )
    print(hls_result.stdout)
    if hls_result.returncode != 0:
        print('ERROR: Vitis HLS synthesis failed')
        print(hls_result.stderr)
    else:
        print('Vitis HLS synthesis completed successfully.')

    # Step 2: Generate bitstream with Vivado
    print('--- Running Vivado bitstream generation ---')
    vivado_result = subprocess.run(
        ['vivado', '-mode', 'batch', '-source', 'axi_stream_design.tcl'],
        capture_output=True,
        text=True,
        cwd=str(vivado_dir),
    )
    print(vivado_result.stdout)
    if vivado_result.returncode != 0:
        print('ERROR: Vivado bitstream generation failed')
        print(vivado_result.stderr)
    else:
        print('Vivado bitstream generation completed successfully.')